In [1]:

import os
import numpy as np
import pandas as pd
import librosa
import tensorflow as tf
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split


I0000 00:00:1774527731.668238   49408 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1774527731.702050   49408 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1774527732.447295   49408 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [2]:
SR = 32000
DURATION = 5  # seconds
IMG_SIZE = 224
N_MELS = 128
BATCH_SIZE = 16
EPOCHS = 5

In [3]:
def audio_to_mel(path):
    y, sr = librosa.load(path, sr=SR)
    
    # pad or trim
    target_len = SR * DURATION
    if len(y) < target_len:
        y = np.pad(y, (0, target_len - len(y)))
    else:
        y = y[:target_len]
    
    mel = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=N_MELS)
    log_mel = librosa.power_to_db(mel)
    
    # normalize
    log_mel = (log_mel - log_mel.mean()) / (log_mel.std() + 1e-6)
    
    return log_mel

In [4]:
def split_audio(path):
    y, sr = librosa.load(path, sr=SR)
    chunk_size = SR * DURATION
    
    chunks = []
    for i in range(0, len(y), chunk_size):
        chunk = y[i:i+chunk_size]
        
        if len(chunk) < chunk_size:
            chunk = np.pad(chunk, (0, chunk_size - len(chunk)))
        
        chunks.append(chunk)
    
    return chunks

In [5]:
df = pd.read_csv("birdclef-2026/train.csv")

labels = sorted(df['primary_label'].unique())
label_to_idx = {l: i for i, l in enumerate(labels)}
num_classes = len(labels)

In [6]:
num_classes

206

In [8]:
def generator(df):
    for _, row in df.iterrows():
        path = f"birdclef-2026/train_audio/{row['filename']}"
        label = label_to_idx[row['primary_label']]
        
        chunks = split_audio(path)
        
        for chunk in chunks:
            mel = librosa.feature.melspectrogram(y=chunk, sr=SR, n_mels=N_MELS)
            log_mel = librosa.power_to_db(mel)
            
            log_mel = (log_mel - log_mel.mean()) / (log_mel.std() + 1e-6)
            
            img = tf.image.resize(log_mel[..., None], (IMG_SIZE, IMG_SIZE))
            
            y_vec = np.zeros(num_classes)
            y_vec[label] = 1
            
            yield img, y_vec

In [9]:
output_signature = (
    tf.TensorSpec(shape=(IMG_SIZE, IMG_SIZE, 1), dtype=tf.float32),
    tf.TensorSpec(shape=(num_classes,), dtype=tf.float32),
)

dataset = tf.data.Dataset.from_generator(
    lambda: generator(df),
    output_signature=output_signature
)

dataset = dataset.shuffle(512).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

W0000 00:00:1774527931.413343   49408 gpu_device.cc:2365] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


In [10]:
from tensorflow.keras import layers, models

def build_model():
    inputs = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 1))
    
    x = layers.Conv2D(32, 3, activation='relu')(inputs)
    x = layers.MaxPooling2D()(x)
    
    x = layers.Conv2D(64, 3, activation='relu')(x)
    x = layers.MaxPooling2D()(x)
    
    x = layers.Conv2D(128, 3, activation='relu')(x)
    x = layers.GlobalAveragePooling2D()(x)
    
    x = layers.Dense(128, activation='relu')(x)
    outputs = layers.Dense(num_classes, activation='sigmoid')(x)
    
    return models.Model(inputs, outputs)

model = build_model()

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 224, 224, 1)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 222, 222, 32)   │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 109, 109, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 54, 54, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 52, 52, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 128)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 206)            │        26,574 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 135,758 (530.30 KB)

 Trainable params: 135,758 (530.30 KB)

 Non-trainable params: 0 (0.00 B)

In [11]:
model.fit(dataset, epochs=EPOCHS)

Epoch 1/5


I0000 00:00:1774527999.086099   49999 generator_dataset_op.cc:213] Memory patch applied: M_TRIM_THRESHOLD=128 kb was set.
I0000 00:00:1774528009.083159   49999 shuffle_dataset_op.cc:453] ShuffleDatasetV3:2: Filling up shuffle buffer (this may take a while): 398 of 512
I0000 00:00:1774528010.352889   49999 shuffle_dataset_op.cc:483] Shuffle buffer filled.


16621/16621 ━━━━━━━━━━━━━━━━━━━━ 4290s 257ms/step - accuracy: 0.5749 - loss: 0.0135
Epoch 2/5


/home/enky-yy/anaconda3/lib/python3.13/site-packages/keras/src/trainers/epoch_iterator.py:164: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


16621/16621 ━━━━━━━━━━━━━━━━━━━━ 4377s 263ms/step - accuracy: 0.1317 - loss: 0.0309
Epoch 3/5
  164/16621 ━━━━━━━━━━━━━━━━━━━━ 1:13:17 267ms/step - accuracy: 0.0000e+00 - loss: 0.0522

KeyboardInterrupt: 